
# Portfolio Health Analysis

**Case:** PH – Portfolio Health

**Your Mission:** Investigate portfolio health crisis at EduFin using SQL.

---

**Situation:** Portfolio crisis confirmed. Board knows something is wrong – they need you to quantify WHAT.

**Your Assignment:** Portfolio health assessment using SQL investigation.

---

# Dataset Progression

### PH – Portfolio Health

Dataset used:

`workspace.edufin_static`

Purpose:

* Build baseline portfolio understanding
* Execute first portfolio assessment case
* No dataset switching in this case

---

# Deliverables: 
- Queries 1A-1E as specified below
- Executive summary interpreting findings
- WHY Gate answers (business reasoning)
- Scale comparison observations (5K vs 500K)

---
# BEFORE YOU START

1. Run Data Validation Check (Cell 2)
2. Solve queries sequentially
3. Use `workspace.edufin_static`
4. Complete all deliverables before submission

---

In [0]:
%sql
-- DATA VALIDATION CHECK
-- Run this cell FIRST to verify your environment is ready
-- This is NOT a graded query - just a system check

SELECT 
  COUNT(*) AS total_loans,
  COUNT(DISTINCT loan_status) AS status_types,
  COUNT(DISTINCT customer_id) AS unique_customers,
  MIN(loan_amount) AS min_loan,
  MAX(loan_amount) AS max_loan
FROM workspace.edufin_small.loans;

-- Expected: ~5000 loans, 4 status types, ~2400 customers
-- If you see 0 loans or errors, contact support before proceeding

total_loans,status_types,unique_customers,min_loan,max_loan
5000,4,3000,150000.0,986731.5


### How is the Portfolio distributed accross the statuses?

In [0]:
%sql
-- QUERY 1A: Portfolio Status Distribution
--
-- BUSINESS PURPOSE:
-- How is the portfolio distributed across statuses?
-- Are most loans healthy (Active/Closed) or problematic (Defaulted/Overdue)?
-- This establishes the baseline: Is this a localized issue or portfolio-wide crisis?
--
-- WHAT TO CALCULATE:
-- Show how the portfolio is distributed across different statuses.
-- Include both counts and percentages for each status.
-- Sort by largest category first to show the dominant pattern immediately.
--
-- EXPECTED INSIGHT:
-- If Defaulted + Overdue > 15% combined, this is a systemic crisis, not isolated incidents.

-- Write your query below:


SELECT * FROM  workspace.edufin_small.loans LIMIT 5;

loan_id,customer_id,institution_id,loan_amount,loan_status,interest_rate,loan_tenure_months,application_date,disbursement_date,maturity_date,emi_amount,purpose_of_loan
1,2440,2954,190607.125,Defaulted,11.39000034,84,2021-12-08,2021-12-23,2028-11-16,3302.540039,Living Expenses
2,2440,4741,425798.375,Active,14.43999958,48,2022-01-01,2022-01-11,2025-12-21,11728.73047,Course Fees + Living
3,2440,902,318341.25,Defaulted,11.64999962,96,2023-03-06,2023-04-12,2031-03-01,5112.870117,Course Fees + Living
4,4195,4724,327505.75,Active,10.18999958,48,2024-12-20,2025-01-31,2029-01-10,8336.240234,Course Fees + Living
5,4195,244,641567.8125,Active,11.68999958,84,2023-11-16,2023-12-10,2030-11-03,11219.62012,Hostel Fees


### How is Portfolio distributed accross the statuses?

In [0]:
%sql
--  How is the portfolio distributed across statuses?
WITH loan_dist AS (
SELECT 
     COUNT(*) AS total_loans,
     loan_status
     FROM edufin_small.loans
     GROUP BY loan_status
 ),
 loan_total AS (
     SELECT 
            SUM(total_loans) AS total_loans1
     FROM loan_dist
 )
 SELECT 
    d.loan_status, d.total_loans,
     concat(ROUND(( (d.total_loans*100) /  t.total_loans1),2),'%') AS percentage_distribution 
FROM loan_dist d
    CROSS  JOIN loan_total t
ORDER BY percentage_distribution DESC; 


---FINDINGS
---Active loan_status holds the highest percentage of loans(79.3%)
-----Defaulted 11.8%
----Overdue 5.46%
-----Closed 3.44%

loan_status,total_loans,percentage_distribution
Active,3965,79.3%
Overdue,273,5.46%
Closed,172,3.44%
Defaulted,590,11.8%


### Findings
- Active loan_status (79.3%)
- Defaulted 11.8%
- Overdue 5.46%
- Closed 3.44%

#### Are most loans healthy (Active/Closed) or problematic (Defaulted/Overdue)?


In [0]:
%sql
WITH loan_dist AS (
		SELECT 
			COUNT(*) AS total_loans,
			loan_status
			FROM edufin_small.loans
			GROUP BY loan_status
		),
		loan_total AS(
		SELECT 
				SUM(total_loans) AS total_loans
				FROM loan_dist
		),
		 health_status1 AS(
		SELECT 
			CASE 
				WHEN loan_status IN ('Active', 'Closed') THEN 'Healthy'
				WHEN loan_status IN ('Overdue', 'Defaulted') THEN 'Problematic'
				ELSE 'Unknown'
			END AS health_status,
            SUM(total_loans) AS total_loans
			FROM loan_dist
					GROUP BY
					CASE 
							WHEN loan_status IN ('Active', 'Closed') THEN 'Healthy'
				WHEN loan_status IN ('Overdue', 'Defaulted') THEN 'Problematic'
				ELSE 'Unknown'
				END
				)
				SELECT 
			  h.health_status,
			h.total_loans,
				CONCAT((h.total_loans * 100 / SUM(l.total_loans)),'%')AS Percentage_distribution
				FROM health_status1 h
				CROSS JOIN loan_total l
				GROUP BY health_status, l.total_loans, h.total_loans;

health_status,total_loans,Percentage_distribution
Problematic,863,17.26%
Healthy,4137,82.74%


### Findings
- Most Loans are Healthy(Active and Closed) with a percentage of 82.74% 
- Since the defaulted and Overdue > 15 % , Then , this is a systematic crisis.

In [0]:
%sql
-- QUERY 1B: Portfolio Scale Metrics
--
-- BUSINESS PURPOSE:
-- What's the scale of operations? Is this a ₹50 Cr portfolio or ₹500 Cr portfolio?
-- Are we talking about 5,000 customers or 50,000?
-- Scale determines whether a ₹12 Cr discrepancy is catastrophic or manageable.
--
-- WHAT TO CALCULATE:
-- Establish the scale of operations:
-- How many unique customers do we serve?
-- How many total loans are active in the system?
-- What's the total portfolio value (in Crores)?
-- What's the average loan size (in Lakhs)?
--
-- EXPECTED INSIGHT:
-- If average loan size is ₹5L+ and portfolio is ₹500 Cr+, defaults hitting ₹12 Cr signals high-value customer concentration risk.

-- Write your query below:


In [0]:
%sql
-- QUERY 1C: Portfolio Risk Metrics
--
-- BUSINESS PURPOSE:
-- Management wants to understand overall portfolio risk and health.
-- How many loans are healthy, how many are already defaulted, and how many are currently at risk?
--
-- WHAT TO CALCULATE:
-- Show overall portfolio metrics including:
-- Total loan volume
-- Healthy loan count
-- Defaulted loan count
-- Overdue loan count
--
-- Also calculate:
-- Overall default percentage
-- Overall portfolio-at-risk percentage
-- (At-risk means loans already failed OR showing payment stress)
--
-- EXPECTED INSIGHT:
-- Higher default percentage indicates weakening portfolio quality.
-- Higher at-risk percentage means more loans may move into default in future.
-- This helps identify whether the portfolio is stable or entering risk phase.
--
-- Write your query below:

In [0]:
%sql
-- QUERY 1D: Financial Impact by Status
--
-- BUSINESS PURPOSE:
-- Decision-makers think in Crores, not loan counts.
-- "500 defaulted loans" is abstract. "₹8.5 Cr in defaulted loans" is real money.
-- This query translates operational numbers into financial language.
--
-- WHAT TO CALCULATE:
-- Show the financial value (in Crores) locked in each status.
-- How much money is in Active loans (earning interest)?
-- How much is in Defaulted loans (write-off candidates)?
-- How much is in Overdue loans (recovery pipeline)?
-- What's the combined "At Risk Value" (Defaulted + Overdue)?
--
-- EXPECTED INSIGHT:
-- If ₹12 Cr is at risk out of ₹500 Cr portfolio, that's 2.4% - manageable.
-- If ₹12 Cr is at risk out of ₹50 Cr portfolio, that's 24% - serious threat needing immediate action.

-- Write your query below:


In [0]:
%sql
-- QUERY 1E: Executive Dashboard
--
-- BUSINESS PURPOSE:
-- Executives need ONE number that says "We're fine" or "We're in trouble."
-- This query consolidates all previous insights into a single executive-ready view with automated risk classification.
-- It's your "executive summary in a single row" - everything needed to make a go/no-go decision.
--
-- WHAT TO CALCULATE:
-- One comprehensive row showing:
-- All status counts and their percentages
-- Total portfolio financial metrics (values in Crores)
-- Automated health classification (HEALTHY / MODERATE RISK / HIGH RISK / CRITICAL) based on default rate thresholds
--
-- EXPECTED INSIGHT:
-- If health_status = "CRITICAL" and default_rate > 15%, immediate action required (capital raise, lending freeze, recovery acceleration).
-- If health_status = "MODERATE RISK" and default_rate = 8%, monitoring plan needed, not panic.
--
-- After running: Fill in the Executive Summary template below (Cell 8).

-- Write your query below:


## Executive Summary (Query 1E Interpretation)

**Based on your Query 1E output, answer these:**

---

**1. Portfolio Health Status**

What health classification did Query 1E return (HEALTHY / MODERATE RISK / HIGH RISK / CRITICAL)? Does this match what you expected from Queries 1A-1D, or is there a surprise?



**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

**2. Default Rate Analysis**

Your Query 1E shows a default rate of ___%. Is this acceptable for a fintech lender? Compare this to the 5%, 10%, 15% thresholds defined in Cell 1. What does this percentage actually mean for the business?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

**3. Financial Risk Exposure**

Query 1E shows ₹___ Cr at risk (Defaulted + Overdue combined). Out of a total portfolio of ₹___ Cr, this represents ___% of the portfolio. Is this a liquidity crisis, a capital adequacy crisis, or a recoverable situation?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

**4. Recommended Action**

Based on the health status, default rate, and financial exposure above, what should be done in the next 48 hours? Be specific: freeze lending, accelerate recovery, raise capital, or continue monitoring?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

## INVESTIGATION SUMMARY

**Write a 3-4 sentence executive summary synthesizing your complete analysis:**

**Example format:**

*"Portfolio shows 12.4% default rate (MODERATE RISK) with ₹8.7 Cr at risk out of ₹70 Cr total. Risk concentrates in large loans (>₹6L) at 18% default rate. Root cause: lending policy failure post-Jan 2024 migration. Immediate action: freeze large loan approvals, launch ₹8.7 Cr recovery task force, audit migration data."*

---

**Your Summary (use your actual numbers):**







---

### Revision History

**Track your daily refinements - delete the example row, keep only your actual changes:**

| Day | What Changed |
|-----|-------------|
| Day1 | First submission |
| Day2 | [EXAMPLE] Fixed Query 1C risk bands, corrected percentage formatting [DELETE THIS ROW] |
| Day3 | |
| Day4 | |
| Day5 | |

## WHY GATE - Business Reasoning

### Question 1: At-Risk Definition

Your Query 1D calculates "Total At Risk Value". Did you include only Defaulted loans, or both Defaulted + Overdue? Explain why your choice is the correct business definition of "at risk" for a fintech lender in a crisis situation.



**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

### Question 2: Crisis Classification

Look at your Query 1A (status distribution) and Query 1E (health classification). Which specific metric from these queries proves this is a systemic portfolio issue versus a temporary spike in defaults? What number would you cite to justify calling this a "crisis"?

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

### Question 3: Executive KPI Prioritization

If only one metric from this analysis had to be presented to leadership, which metric would you choose and why? Explain how this metric helps leadership evaluate portfolio health and business risk.

**Analytical Findings:** _Document your interpretation and business insight below_

**Observation:**

## SUBMISSION READINESS CHECK

**Before submitting, verify:**

✓ Data Validation (Cell 2) passed

✓ All 5 SQL queries (Cells 3-7) executed without errors

✓ Executive Summary (Cell 8) - all 4 questions answered

✓ Investigation Summary (Cell 9) - synthesis paragraph written, revision history updated

✓ WHY Gate (Cell 10) - all 3 questions answered with reasoning

✓ Financial values in Crores format (₹X.XX Cr)

✓ Percentages show 2 decimals (e.g., 11.78%)

✓ Deleted example rows from Revision History table

---

## SUBMISSION PROCESS

**Step 1:** Run Cell 12 below
- Enter your name and select day
- Get your submission filename and form link

**Step 2:** Download this notebook
- File → Export → Ipython notebook
- Rename the downloaded file as shown in Cell 12 output

**Step 3:** Submit via Google Form
- Use the form link from Cell 12 output
- Upload your ipynb file
- Note your entry number from form confirmation (tracking ID for feedback)

**Step 4:** Wait for feedback
- Expect feedback within 24 hours via WhatsApp

---

**Reminder:** Partial submissions accepted. If only 3 of 5 queries work, submit what you have with documentation of where you got stuck.


In [0]:
import re

FORM_LINK = "https://forms.gle/7xWVjeLRahSEMYXD9"

dbutils.widgets.removeAll()
dbutils.widgets.text("Name", "")
dbutils.widgets.text("EnrollmentID", "")
dbutils.widgets.dropdown("day", "Day1", ["Day1", "Day2", "Day3"])

name = dbutils.widgets.get("Name")
batch = dbutils.widgets.get("EnrollmentID")
day = dbutils.widgets.get("day")

if not name or not batch:
    print("⚠️  Enter your name and BatchNo above, then run this cell")
else:
    clean_name = re.sub(r'[^a-zA-Z0-9]', '', name.lower())
    filename = f"{clean_name}_{batch}_{day}.ipynb"
    
    print("File -> Export -> IPython notebook")
    print(f"Rename to: {filename}")
    print(f"Submit the file in the form link: {FORM_LINK}")
    print("\nPlease await for submission review.")

⚠️  Enter your name and BatchNo above, then run this cell
